# 🎯 Landmark Detection - Inference Demo
## Load Model and Make Predictions

---


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gradio as gr
import json
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROJECT_ROOT = Path('../').resolve()
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
DATA_DIR = PROJECT_ROOT / 'data'

### 2. Load Model

In [ ]:
# Model definition
class LandmarkClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
        self.num_classes = num_classes
    
    def forward(self, x):
        return self.backbone(x)

# Load checkpoint
checkpoint_path = CHECKPOINT_DIR / 'final_model.pth'

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    NUM_CLASSES = checkpoint.get('num_classes', 500)
    
    model = LandmarkClassifier(num_classes=NUM_CLASSES, pretrained=False, dropout=0.3)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f'Model loaded: {NUM_CLASSES} classes')
    print(f'Best val accuracy: {checkpoint.get("best_val_acc", "N/A")}')

In [ ]:
else:
    print('No trained model found!')
    print('Train a model first with:')
    print('  python landmark_detection.py --mode train')
    print('Or run 04_Training.ipynb')

### 3. Inference Pipeline

In [ ]:
# Transforms
IMAGE_SIZE = 224

inference_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load landmark names (if available)
landmark_names = {i: f'Landmark_{i}' for i in range(NUM_CLASSES)}

if (DATA_DIR / 'landmark_mapping.csv').exists():
    mapping = pd.read_csv(DATA_DIR / 'landmark_mapping.csv')
    for _, row in mapping.iterrows():
        landmark_names[row['class_idx']] = f'Landmark_{int(row["landmark_id"])}'

print(f'Landmark names loaded: {len(landmark_names)} classes')

In [ ]:
def predict_landmark(image_path, top_k=5):
    """
    Predict landmark from image file.
    
    Returns:
        prediction (str): Predicted landmark name
        confidence (float): Confidence score
        top_predictions (dict): Top-K predictions with probabilities
    """
    image = Image.open(image_path).convert('RGB')
    img_tensor = inference_transforms(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, dim=1)
    
    pred_class = pred.item()
    confidence = conf.item()
    landmark_name = landmark_names.get(pred_class, f'Landmark_{pred_class}')
    
    # Get top-K predictions
    top_k_probs, top_k_indices = torch.topk(probs, min(top_k, NUM_CLASSES), dim=1)
    top_predictions = {}
    for prob, idx in zip(top_k_probs[0], top_k_indices[0]):
        name = landmark_names.get(idx.item(), f'Landmark_{idx.item()}')
        top_predictions[name] = f'{prob.item()*100:.2f}%'
    
    return f'{landmark_name} ({confidence*100:.1f}%)', {landmark_name: f'{confidence*100:.2f}%', **top_predictions}, image


def predict_image_pil(image):
    """Predict from PIL image."""
    img_tensor = inference_transforms(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, dim=1)
    
    return pred.item(), conf.item()

### 4. Test with Sample Images

In [ ]:
# Test on a sample image if available
IMAGES_DIR = PROJECT_ROOT / 'images_000'

if IMAGES_DIR.exists():
    sample_images = list(IMAGES_DIR.glob('*.jpg'))[:4]
    
    if sample_images:
        print(f'Testing on {len(sample_images)} sample images...')
        print('='*60)
        
        for img_path in sample_images:
            image = Image.open(img_path).convert('RGB')
            img_tensor = inference_transforms(image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = model(img_tensor)
                probs = torch.softmax(outputs, dim=1)
                conf, pred = torch.max(probs, dim=1)
            
            pred_class = pred.item()
            landmark = landmark_names.get(pred_class, f'Class_{pred_class}')
            print(f'{img_path.name}: {landmark} ({conf.item()*100:.1f}%)')
    else:
        print('No images downloaded yet.')

### 5. Gradio Interface

In [ ]:
try:
    def gradio_predict(image):
        if image is None:
            return 'Please upload an image', {}
        
        pred, conf, _ = predict_landmark(image)
        return pred, conf
    
    interface = gr.Interface(
        fn=gradio_predict,
        inputs=gr.Image(type='pil', label='Upload Landmark Image'),
        outputs=[gr.Label(num_top_classes=5, label='Predictions'), gr.Label(num_top_classes=5, label='Details')],
        title='\U0001F3DB\uFE0F Landmark Detection',
        description='Upload an image to identify famous landmarks!',
        examples=[
            ['sample1.jpg'],
            ['sample2.jpg'],
        ] if IMAGES_DIR.exists() and list(IMAGES_DIR.glob('*.jpg')) else None
    )
    
    print('Gradio interface ready!')
    print('Run: interface.launch() to start the web server')
    
    # Uncomment to launch:
    # interface.launch(share=True)

except ImportError:
    print('Gradio not installed. Install with: pip install gradio')

### 6. Export to ONNX

In [ ]:
# Export model to ONNX
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

if checkpoint_path.exists():
    onnx_path = OUTPUT_DIR / 'landmark_model.onnx'
    
    model.eval()
    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_path),
        export_params=True,
        opset_version=11,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f'Model exported to ONNX: {onnx_path}')

### 7. Summary

In [ ]:
print(f"""
INFERENCE SETUP COMPLETE!

Usage:
  1. python landmark_detection.py --mode inference --image path/to/image.jpg
  2. Or run this notebook and use the prediction functions
  3. Launch Gradio: interface.launch(share=True)

Files:
  - Model: {CHECKPOINT_DIR / 'final_model.pth'}
  - ONNX: {OUTPUT_DIR / 'landmark_model.onnx'}
""")